<a href="https://colab.research.google.com/github/ankit-rathi/Quantvesting_v2/blob/main/myStocks_Portfolio_Analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
'''
!pip install pandas==2.2.2 -qq
!pip install pyxirr -qq
!pip install ta -qq
!pip install yfinance==0.2.59 -qq
'''

'\n!pip install pandas==2.2.2 -qq\n!pip install pyxirr -qq\n!pip install ta -qq\n!pip install yfinance==0.2.59 -qq\n'

In [2]:
# import libraries

import numpy as np
import pandas as pd
import datetime
import warnings
warnings.filterwarnings('ignore')
import yfinance as yfin
import ta
import matplotlib.pyplot as plt
import requests

In [3]:
# notebook start time
import time
import datetime
import pytz

start_time = time.time()

# function to print date timestamp
def print_date_time():
  tz_NY = pytz.timezone('Asia/Kolkata')
  datetime_NY = datetime.datetime.now(tz_NY)
  print("Run date time (IST):", datetime_NY.strftime("%Y-%m-%d %H:%M:%S"))

In [4]:
# function to get booked and reserved amount
def get_amt():
  # fetch the JSON data from the URL
  url = "https://raw.githubusercontent.com/ankit-rathi/Tradevesting_v1/main/data/myPortfolioAmts.json"
  response = requests.get(url)
  pf_amts = response.json()  # parse the JSON data

  # extract values from the JSON
  py_booked_amt_dm = pf_amts["py_booked_amt_dm"]
  py_booked_amt_sv = pf_amts["py_booked_amt_sv"]
  cy_booked_amt_dm = pf_amts["cy_booked_amt_dm"]
  cy_booked_amt_sv = pf_amts["cy_booked_amt_sv"]
  reserve_amt_dm = pf_amts["reserve_amt_dm"]
  reserve_amt_sv = pf_amts["reserve_amt_sv"]

  # perform calculations
  py_booked_amt = py_booked_amt_dm + py_booked_amt_sv
  cy_booked_amt = cy_booked_amt_dm + cy_booked_amt_sv
  reserve_amt = reserve_amt_dm + reserve_amt_sv
  total_booked_amt = py_booked_amt + cy_booked_amt

  return total_booked_amt, reserve_amt, cy_booked_amt, py_booked_amt

gen_output = 0

In [5]:
# function to format the amount
def format_amt(number):
    abs_number = abs(number)

    if abs_number >= 1_00_00_000:  # Crores
        formatted_amt = f"{number / 1_00_00_000:.2f} C"
    elif abs_number >= 1_00_000:  # Lakhs
        formatted_amt = f"{number / 1_00_000:.2f} L"
    elif abs_number >= 1_000:  # Thousands
        formatted_amt = f"{number / 1_000:.2f} K"
    else:
        formatted_amt = f"{number:.2f}"

    return formatted_amt

# function to set start and end date
def get_start_end_date():
  start_date = (datetime.date.today() + datetime.timedelta(days=-365)).strftime('%Y-%m-%d')
  end_date = (datetime.date.today() + datetime.timedelta(days=1)).strftime('%Y-%m-%d')
  return start_date, end_date

# function to fetch my portfolio csv
def get_mypfs_df():
  mypfs_df = pd.read_csv('https://raw.githubusercontent.com/ankit-rathi/Tradevesting_v1/main/data/myPortfolioStocks.csv')
  return mypfs_df

# function to fetch my prospects csv
def get_mypps_df():
  mypps_df = pd.read_csv('https://raw.githubusercontent.com/ankit-rathi/Tradevesting_v1/main/data/myProspectsScrips.csv')
  return mypps_df

# function to fetch screener data
def get_myscreen_df():
  myscreen_df = pd.read_csv('https://raw.githubusercontent.com/ankit-rathi/Tradevesting_v1/main/data/myScreenerDB.csv')
  return myscreen_df

# function to fetch momentum data
def get_myinvmt_df():
  myinvmt_df = pd.read_csv('https://raw.githubusercontent.com/ankit-rathi/Tradevesting_v1/main/data/myInvestments.csv')
  return myinvmt_df

# function to get the stock ids
def get_stock_ids(df_pf):
  stock_n100 = df_pf['Symbol'].unique()

  exclude = ['CADILAHC','MMTC', 'MASFIN']

  stock_ids = df_pf[~df_pf['Symbol'].isin(exclude) ]['Symbol'].unique()

  #mypf = mypf[mypf.Forecast.notnull()]
  #stock_ids = mypf['Symbol'].unique()

  #stock_ids.sort()
  return stock_ids

# get features from screener data
def get_screener_features():
  myscreen_df = get_myscreen_df()
  cols = ['Symbol', 'EPS', 'MedPE', 'ROCE%', 'ROE%', 'CapType']
  return myscreen_df[cols]

# get relative strength
def get_relative_strength(stock_list):

    # Dictionary to store stock tickers and their corresponding percentage price change
    stock_changes = {}

    # Loop through each stock and fetch its price data
    for stock in stock_list:
        try:
            # Download the stock data for the given date range
            stock_data = yfin.Ticker(stock + '.NS').history(period='1mo', interval='1d')[map(str.title, ['open', 'close', 'low', 'high', 'volume'])]

            # Calculate the percentage change for the stock
            if len(stock_data) > 0:
                start_price = stock_data['Close'].iloc[0]
                end_price = stock_data['Close'].iloc[-1]
                percent_change = round(((end_price - start_price) / start_price) * 100, 2)
                stock_changes[stock] = percent_change
            else:
                stock_changes[stock] = np.nan  # If no data is available, set to NaN

        except Exception as e:
            print(f"Error fetching data for {stock}: {e}")
            stock_changes[stock] = np.nan

    # Create a DataFrame with stock tickers and their percentage changes
    df = pd.DataFrame(list(stock_changes.items()), columns=['Symbol', 'Percent_Change'])

    # Drop any stocks with missing data (NaN values)
    df = df.dropna()

    # Calculate the percentile rank based on percentage change
    df['RSP'] = round(df['Percent_Change'].rank(pct=True) * 100, 2)
    cols = ['Symbol', 'RSP']
    df = df[cols]
    # Sort by percentile rank (optional)
    df = df.sort_values(by='RSP', ascending=False).reset_index(drop=True)

    return df


In [6]:
# function to get stock technicals
def stock_prec_dev(stock_symbol):
    #stock_symbol = 'ULTRACEMCO.NS'
    short_window = 20
    mid_window = 50
    long_window = 200
    moving_avg = 'SMA'

    start = datetime.datetime(*map(int, start_date.split('-')))
    end = datetime.datetime(*map(int, end_date.split('-')))

    stock_df = yfin.Ticker(stock_symbol + '.NS').history(period='max', interval='1d')[map(str.title, ['open', 'close', 'low', 'high', 'volume'])]

    stock_df['Max'] = round(max(stock_df['Close']),2)
    stock_df = stock_df[(stock_df.index <= end_date) & (stock_df.index >= start_date)]
    stock_df['200_SMA'] = round(stock_df['Close'].rolling(window = 200, min_periods = 1).mean(),0)
    stock_df['Dev%_200'] = round((stock_df['Close'] - stock_df['200_SMA'])*100/stock_df['200_SMA'],2)
    stock_df.dropna(axis = 0, inplace = True) # remove any null rows

    stock_df['20_SMA'] = stock_df['Close'].rolling(window=20).mean()
    stock_df['50_SMA'] = stock_df['Close'].rolling(window=50).mean()
    stock_df['Symbol'] = stock_symbol

    stock_df['Close'] = round(stock_df['Close'],2)
    stock_df['Min'] = round(min(stock_df['Close']),2)
    stock_df['RSI_14'] = ta.momentum.RSIIndicator(close=stock_df['Close'], window=14).rsi()
    stock_df['RSI_14'] = round(stock_df['RSI_14'],0)
    stock_df['Prev_Close'] = stock_df['Close'].shift(1)
    stock_df.drop(['Open', 'Low', 'High', 'Volume'], axis=1, inplace=True)
    stock_df = stock_df.tail(1)

    max_SMA = max(stock_df['20_SMA'].item(), stock_df['50_SMA'].item(), stock_df['200_SMA'].item())
    min_SMA = min(stock_df['20_SMA'].item(), stock_df['50_SMA'].item(), stock_df['200_SMA'].item())
    ABS_Spread = max_SMA - min_SMA
    stock_df['Spread%'] = round((ABS_Spread / stock_df['200_SMA'].item()) * 100,2)

    return stock_df

# function to compute stock attributes
def get_common_features(stock_ids, df_mypf):

  df_prec_dev = pd.DataFrame()
  df_tmp = get_screener_features()
  df_rs = get_relative_strength(stock_ids)

  for stock_id in stock_ids:
      tmp = stock_prec_dev(stock_id)
      tmp = tmp.reset_index()
      df_prec_dev = pd.concat([df_prec_dev, tmp], ignore_index = True)
  df_prec_dev = pd.merge(df_prec_dev, df_mypf, on= 'Symbol')
  df_prec_dev = pd.merge(df_prec_dev, df_tmp, on= 'Symbol', how='left')
  df_prec_dev = pd.merge(df_prec_dev, df_rs, on= 'Symbol', how='left')
  #print(df_prec_dev.columns)
  df_prec_dev['Curr_PE'] = round(df_prec_dev['Close']/df_prec_dev['EPS'],1)
  df_prec_dev['Dev%_PE'] = round((df_prec_dev['Curr_PE'] - df_prec_dev['MedPE'])*100/df_prec_dev['MedPE'],2)
  df_prec_dev['Conviction'] = df_prec_dev['Conviction'] + '-' + df_prec_dev['CapType']
  return df_prec_dev

# function to arrange stock features
def arrange_features(df_stocks, common_cols, diff_cols):
  df_stocks_common = df_stocks[common_cols].drop_duplicates()
  df_stocks_diff = df_stocks[diff_cols]
  df_stocks_diff['Investment'] = df_stocks_diff['AvgCost'] * df_stocks_diff['Shares']
  df_stocks_diff = df_stocks_diff.groupby(['Symbol'])[['Shares', 'Investment']].aggregate(['sum']).reset_index()
  df_stocks_diff.columns = ['Symbol', 'Shares', 'Investment']
  df_stocks_diff['AvgCost'] = round(df_stocks_diff['Investment']/df_stocks_diff['Shares'],2)
  df_stocks = pd.merge(df_stocks_diff, df_stocks_common, on='Symbol')
  return df_stocks

# plot fact distribution across dimension
def plot_pie_chart(df, dimension, fact):
  # grouping the data by category and calculating the sum of fact for each type
  grouped_data = df.groupby(dimension)[fact].sum()

  # sorting the grouped data in descending order
  grouped_data = grouped_data.sort_values(ascending=False)

  # creating a pie chart
  grouped_data.plot.pie(autopct='%1.1f%%', startangle=90, figsize=(6, 6))

  # adding a title and displaying the plot
  plt.title(f'{dimension} {fact} Distribution')
  plt.ylabel('')  # To hide the y-label
  plt.show()

In [7]:
# function to get portfolio features

def get_portfolio_features(df_common_features):

  print_date_time()
  print('=================== PORTFOLIO QUALITY ====================')

  #df_common_features["Target"] = df_common_features["Target"].fillna(df_common_features["Max"])
  df_common_features['NTT'] = np.where(df_common_features['Strategy']== 'NTT', df_common_features["Target"], df_common_features['Max'])
  df_common_features['LTT'] = np.where(df_common_features['Strategy']== 'BTT', df_common_features["Target"], df_common_features['Max'])
  df_common_features['BOL'] = df_common_features['Min']

  tmp_df = df_common_features[df_common_features['Symbol'].isin(mypf_df[mypf_df['InPortfolio'] != 'NA'].Symbol.values)]
  print('qualified stocks: '+str(len(tmp_df['Symbol'].unique())))
  tmp_df1 = tmp_df[tmp_df['LatestQtr'] == 1]
  print('with latest results: '+str(len(tmp_df1['Symbol'].unique())))
  tmp_df1 = tmp_df1[tmp_df1['StarStock'] == 1]
  print('still star stocks: '+str(len(tmp_df1['Symbol'].unique())))
  tmp_df['Investment'] = tmp_df['AvgCost'] * tmp_df['Shares']
  tmp_df['Current'] = round(tmp_df['Close'] * tmp_df['Shares'],0)
  tmp_df['Previous'] = tmp_df['Prev_Close'] * tmp_df['Shares']
  tmp_df['EstimatedST'] = tmp_df['NTT'] * tmp_df['Shares']
  tmp_df['EstimatedLT'] = tmp_df['LTT'] * tmp_df['Shares']
  tmp_df['Current P/L'] = round((tmp_df['Current'] - tmp_df['Investment']),0)
  tmp_df['Today P/L%'] = round((tmp_df['Current'] - tmp_df['Previous'])*100/tmp_df['Previous'],2)
  tmp_df['Current P/L%'] = round((tmp_df['Current'] - tmp_df['Investment'])*100/tmp_df['Investment'],2)
  tmp_df['EstimatedST P/L%'] = round((tmp_df['EstimatedST'] - tmp_df['Investment'])*100/tmp_df['Investment'],2)
  tmp_df['EstimatedLT P/L%'] = round((tmp_df['EstimatedLT'] - tmp_df['Investment'])*100/tmp_df['Investment'],2)
  tmp_df['NTT%'] = round((tmp_df['NTT'] - tmp_df['Close'])*100/tmp_df['Close'],2)
  tmp_df['LTT%'] = round((tmp_df['LTT'] - tmp_df['Close'])*100/tmp_df['Close'],2)
  tmp_df['Gained%'] = round((tmp_df['Close'] - tmp_df['BOL'])*100/tmp_df['BOL'],2)
  investment = round(sum(tmp_df['AvgCost']*tmp_df['Shares']),0)
  current = round(sum(tmp_df['Close']*tmp_df['Shares']),0)
  tmp_df['InitAlloc%'] = round(tmp_df['Investment']*100/investment,2)
  tmp_df['CurrAlloc%'] = round(tmp_df['Current']*100/current,2)
  tmp_df['FTT'] = tmp_df['LTT']
  tmp_df.loc[tmp_df['Strategy'] == 'NTT', 'FTT'] = tmp_df['NTT']
  tmp_df['FTT%'] = tmp_df['LTT%']
  tmp_df.loc[tmp_df['Strategy'] == 'NTT', 'FTT%'] = tmp_df['NTT%']
  tmp_df['FTT Amt'] = round(tmp_df['FTT%'] * tmp_df['Current']/100,0)
  tmp_df['OTT%'] = round((tmp_df['FTT'] - tmp_df['AvgCost'])*100/tmp_df['AvgCost'],2)
  tmp_df['RRR Ind'] = round(tmp_df['Current P/L']/tmp_df['FTT Amt'],2)
  tmp_df['Risk Ind'] = round(tmp_df['Current P/L%']*tmp_df['CurrAlloc%'],0)

  return tmp_df

# function to print portfolio features
def print_portfolio_stats(df_portfolio_features, myinvmt_df):
  from pyxirr import xirr

  total_booked_amt, reserve_amt, cy_booked_amt, py_booked_amt = get_amt()

  dates = myinvmt_df['Date'].values
  dates = np.append(dates, datetime.date.today().strftime('%d-%b-%y'))
  investment = myinvmt_df['Investment'].values
  dates= pd.to_datetime(dates)

  current = round(sum(df_portfolio_features['Close']*df_portfolio_features['Shares']),0) + reserve_amt
  investment_xirr = np.append(investment, current)
  cagr = round(xirr(pd.DataFrame({"dates": dates, "amounts": investment_xirr}))*100,2)

  investment = -sum(investment)
  invested = round(sum(df_portfolio_features['AvgCost']*df_portfolio_features['Shares']),0) + reserve_amt
  previous = round(sum(df_portfolio_features['Prev_Close']*df_portfolio_features['Shares']),0) + reserve_amt
  cy_invested = investment + py_booked_amt

  today_pnl_amount = current-previous
  today_pnl_percentage = round((current-previous)*100/previous,2)

  curr_pnl_amount = current-invested
  curr_pnl_percentage = round((curr_pnl_amount)*100/(cy_invested),2)

  cy_pnl_amount = cy_booked_amt + curr_pnl_amount
  cy_pnl_percentage = round((cy_pnl_amount)*100/cy_invested,2)

  overall_pnl_amount = total_booked_amt + curr_pnl_amount
  overall_pnl_percentage = round((overall_pnl_amount)*100/investment,2)

  estimate_st = round(sum(df_portfolio_features['FTT']*df_portfolio_features['Shares']),0)  + reserve_amt
  est_st_pnl_amount = estimate_st-current
  est_st_pnl_percentage = round((est_st_pnl_amount)*100/current,2)

  estimate_lt = round(sum(df_portfolio_features['LTT']*df_portfolio_features['Shares']),0)  + reserve_amt
  est_lt_pnl_amount = estimate_lt-current
  est_lt_pnl_percentage = round((est_lt_pnl_amount)*100/current,2)

  total_profit = round(sum(df_portfolio_features[df_portfolio_features['Current P/L%'] > 0]['Current']) - sum(df_portfolio_features[df_portfolio_features['Current P/L%'] > 0]['Investment']),0)
  total_loss = round(sum(df_portfolio_features[df_portfolio_features['Current P/L%'] < 0]['Current']) - sum(df_portfolio_features[df_portfolio_features['Current P/L%'] < 0]['Investment']),0)

  cy_booked_percentage = round((cy_booked_amt)*100/current,2)
  py_booked_percentage = round((py_booked_amt)*100/investment,2)
  total_booked_percentage = round((total_booked_amt)*100/investment,2)

  print('=================== PORTFOLIO SNAPSHOT ===================')
  print('Deployed Value: ', format_amt(investment))
  print('Current Value:  ', format_amt(current))
  print('CAGR/XIRR %:    ', '{}%'.format(cagr))
  print('Est FTT Value:  ', format_amt(estimate_st))
  print('Future Upside:  ', '{} ({}%)'.format(
      format_amt(est_st_pnl_amount),
      est_st_pnl_percentage
  ))

  print('=================== PERFORMANCE ==========================')
  print('Current PnL:             {} ({}%)'.format(
      format_amt(curr_pnl_amount), curr_pnl_percentage
  ))
  print('Total Booked PnL:        {} ({}%)'.format(
      format_amt(total_booked_amt), total_booked_percentage
  ))
  print('Booked + Current PnL:    {} ({}%)'.format(
      format_amt(overall_pnl_amount), overall_pnl_percentage
  ))
  print('CY Booked PnL:           {} ({}%)'.format(
      format_amt(cy_booked_amt), cy_booked_percentage
  ))
  print('Prev Year Booked PnL:    {} ({}%)'.format(
      format_amt(py_booked_amt), py_booked_percentage
  ))

  print('=================== TODAY / OPERATIONS ===================')
  print('Today PnL:               {} ({}%)'.format(
      format_amt(today_pnl_amount), today_pnl_percentage
  ))
  print('CY Investment:           ', format_amt(cy_invested))
  print('Reserve:                 ', format_amt(reserve_amt))

  print('=================== P&L DETAIL ============================')
  print('Total Profit:            ', format_amt(total_profit))
  print('Total Loss:              ', format_amt(total_loss))
  print('============================================================')

In [8]:
# get start and end date
start_date, end_date = get_start_end_date()

# get portfolio and prospects data
mypfs_df = get_mypfs_df()
mypps_df = get_mypps_df()
myinvmt_df = get_myinvmt_df()

# merge above datasets
mypf_df = pd.merge(mypfs_df, mypps_df, on="Symbol")

# seggregate dm and sv portfolio
dm_pf = mypf_df[mypf_df['InPortfolio'] == 'DM']
sv_pf = mypf_df[mypf_df['InPortfolio'] == 'SV']

# build portfolio stock dataframe
dm_stocks = get_stock_ids(dm_pf)
sv_stocks = get_stock_ids(sv_pf)
df_stocks = pd.concat([dm_pf,sv_pf], ignore_index = True)

# arrange common and diff stock features
common_cols = ['Symbol', 'Target', 'Criteria', 'Strategy', 'CumlRnk', 'LatestQtr', 'StarStock', 'Conviction', 'Category', 'InFolio', 'XIRR', 'MBQ']
diff_cols = ['Symbol', 'AvgCost', 'Shares']
df_stocks = arrange_features(df_stocks, common_cols, diff_cols)

# get common features
stock_ids = df_stocks['Symbol'].values
df_common_features = get_common_features(stock_ids, df_stocks)

df_common_features.reset_index(drop=True, inplace=True)
df_common_features.drop(['Date'], axis=1, inplace=True)
# get and print portfolio features
df_portfolio_features = get_portfolio_features(df_common_features)

print_portfolio_stats(df_portfolio_features, myinvmt_df)

df = df_portfolio_features
#plot_pie_chart(df, 'CapType', 'Current')
list_ox40n = list(df[df['MBQ'].str.contains('OX40N', na=False)]['Symbol'].values)

Run date time (IST): 2026-11-04 22:06:15
=================== PORTFOLIO QUALITY ====================
qualified stocks: 86
with latest results: 86
still star stocks: 46
=================== PORTFOLIO SNAPSHOT ===================
Deployed Value:  1.49 C
Current Value:   1.50 C
CAGR/XIRR %:     0.4%
Est FTT Value:   2.74 C
Future Upside:   1.24 C (82.31%)
=================== PERFORMANCE ==========================
Current PnL:             -46.11 L (-24.03%)
Total Booked PnL:        48.59 L (32.6%)
Booked + Current PnL:    2.47 L (1.66%)
CY Booked PnL:           5.71 L (3.8%)
Prev Year Booked PnL:    42.87 L (28.76%)
=================== TODAY / OPERATIONS ===================
Today PnL:               1.62 L (1.09%)
CY Investment:            1.92 C
Reserve:                  1.90 K
=================== P&L DETAIL ============================
Total Profit:             1.97 L
Total Loss:               -48.09 L


In [9]:
# top 5 near their targets
cols = ['Symbol', 'Today P/L%', 'Current P/L%', 'FTT%', 'OTT%', 'FTT Amt', 'Current P/L', 'Current', 'FTT', 'Dev%_PE', 'RSI_14','Conviction', 'Spread%',  'CumlRnk', 'RRR Ind', 'CurrAlloc%', 'Gained%', 'Criteria', 'Strategy', 'Category']
df_tmp = df_portfolio_features[~df_portfolio_features['Symbol'].isin(['ENRIN','BLUSPRING','DIGITIDE'])]
df_tmp = df_tmp.sort_values(by = 'FTT Amt', ascending=True)
df_tmp[cols].head()

,Symbol,Today P/L%,Current P/L%,FTT%,OTT%,FTT Amt,Current P/L,Current,FTT,Dev%_PE,RSI_14,Conviction,Spread%,CumlRnk,RRR Ind,CurrAlloc%,Gained%,Criteria,Strategy,Category
85,ZYDUSLIFE,-4.23,25.29,8.16,35.51,21426.0,52996.0,262570.0,1285.01,23.89,54.0,H-LC,11.58,120.0,2.47,1.75,38.18,AR,ATH,PHARMA
76,TTKPRESTIG,4.90,-28.87,40.72,0.09,29189.0,-29094.0,71683.0,770.00,-13.37,58.0,M-SC,4.66,250.0,-1.00,0.48,29.02,OX40N,NTT,DURABLES
12,BOSCH-HCIL,0.27,-24.64,32.67,-0.01,31468.0,-31485.0,96320.0,2282.00,752.22,46.0,M-SC,20.87,231.0,-1.00,0.64,63.95,OX40N,BTT,AC
66,SIS,3.06,-11.11,36.42,21.26,35946.0,-12334.0,98698.0,528.00,89.53,47.0,H-SC,8.20,168.0,-0.34,0.66,46.75,OX40N,NTT,MISC
40,ICICIPRULI,0.66,-32.20,27.50,-13.56,36730.0,-63443.0,133562.0,522.35,-56.58,27.0,X-MC,9.01,78.0,-1.73,0.89,0.66,X40,ATH,INSURANCE


In [10]:
# top 5 to accumulate
df_tmp = df_portfolio_features[df_portfolio_features['Conviction'].isin(['X-LC','H-LC'])]
df_tmp = df_tmp[(df_tmp['CumlRnk'] < 40) & (df_tmp['FTT%'] > 20)].sort_values(by = 'CurrAlloc%', ascending=True)
df_tmp[cols].head()

,Symbol,Today P/L%,Current P/L%,FTT%,OTT%,FTT Amt,Current P/L,Current,FTT,Dev%_PE,RSI_14,Conviction,Spread%,CumlRnk,RRR Ind,CurrAlloc%,Gained%,Criteria,Strategy,Category
52,MARUTI,1.06,-12.36,42.73,25.08,112760.0,-37228.0,263890.0,17120.06,-17.84,59.0,X-LC,14.19,33.0,-0.33,1.75,7.39,XR,ATH,AUTO
38,HYUNDAI,1.63,6.71,34.89,43.94,93571.0,16858.0,268190.0,2782.73,10.68,60.0,X-LC,6.24,29.0,0.18,1.78,23.13,X40N,ATH,AUTO
51,M&M,1.85,-7.53,27.04,17.48,75284.0,-22656.0,278419.0,3762.88,-19.50,56.0,X-LC,9.91,25.0,-0.30,1.85,7.66,X40N,ATH,AUTO
35,HDFCLIFE,1.33,-7.76,52.38,40.56,150340.0,-24141.0,287018.0,811.41,-32.80,50.0,X-LC,10.22,31.0,-0.16,1.91,3.68,X40,ATH,INSURANCE
4,ASIANPAINT,-0.57,-12.34,41.52,24.06,120489.0,-40836.0,290196.0,3422.40,-23.14,57.0,X-LC,6.73,30.0,-0.34,1.93,15.26,X40,ATH,PAINTS


In [11]:
# top 5 today
df_tmp = df_portfolio_features.sort_values(by = 'Today P/L%', ascending=False)
df_tmp[cols].head()

,Symbol,Today P/L%,Current P/L%,FTT%,OTT%,FTT Amt,Current P/L,Current,FTT,Dev%_PE,RSI_14,Conviction,Spread%,CumlRnk,RRR Ind,CurrAlloc%,Gained%,Criteria,Strategy,Category
68,SURYODAY,6.58,-16.10,46.64,23.03,70075.0,-28825.0,150246.0,216.00,-30.92,57.0,H-SC,5.30,166.0,-0.41,1.00,27.53,XR,NTT,BANKS
1,ABBOTINDIA,5.12,-9.02,26.81,15.37,43625.0,-16135.0,162720.0,34390.26,-25.92,62.0,X-MC,1.44,58.0,-0.37,1.08,10.22,X40,ATH,PHARMA
76,TTKPRESTIG,4.90,-28.87,40.72,0.09,29189.0,-29094.0,71683.0,770.00,-13.37,58.0,M-SC,4.66,250.0,-1.00,0.48,29.02,OX40N,NTT,DURABLES
25,FINCABLES,4.55,50.97,36.74,106.44,81895.0,75260.0,222904.0,1629.94,10.09,42.0,M-SC,25.86,220.0,0.92,1.48,68.89,OX40N,ATH,CABLES
49,KANSAINER,4.38,-34.81,74.07,13.47,130206.0,-93879.0,175788.0,340.00,-31.79,62.0,H-SC,8.37,162.0,-0.72,1.17,24.25,XY24,NTT,PAINTS


In [12]:
# bottom 5 today
df_tmp = df_portfolio_features.sort_values(by = 'Today P/L%', ascending=True)
df_tmp[cols].head()

,Symbol,Today P/L%,Current P/L%,FTT%,OTT%,FTT Amt,Current P/L,Current,FTT,Dev%_PE,RSI_14,Conviction,Spread%,CumlRnk,RRR Ind,CurrAlloc%,Gained%,Criteria,Strategy,Category
85,ZYDUSLIFE,-4.23,25.29,8.16,35.51,21426.0,52996.0,262570.0,1285.01,23.89,54.0,H-LC,11.58,120.0,2.47,1.75,38.18,AR,ATH,PHARMA
3,ALKYLAMINE,-4.18,-32.30,184.42,92.54,126047.0,-32615.0,68348.0,4520.85,NaN,35.0,NaN,9.31,142.0,-0.26,0.45,29.79,SR,ATH,CHEMICALS
10,BATAINDIA,-2.55,-59.15,231.02,35.20,121415.0,-76114.0,52556.0,2096.00,NaN,49.0,NaN,8.33,91.0,-0.63,0.35,9.21,X40,NTT,FOOTWEAR
72,TCS,-2.52,-26.63,72.76,26.75,260548.0,-129983.0,358092.0,4208.34,-40.62,58.0,X-LC,4.96,2.0,-0.50,2.38,24.20,X40,ATH,IT
33,HCLTECH,-2.37,-3.86,30.75,25.70,106713.0,-13938.0,347035.0,1837.07,-8.41,62.0,X-LC,4.76,8.0,-0.13,2.31,38.46,X40,ATH,IT


In [13]:
# top 5 to exit based on CumlRnk
df_tmp = df_portfolio_features[~df_portfolio_features['Conviction'].isin(['X-LC','H-LC','X-MC','X-SC'])]
df_tmp = df_tmp[(df_tmp['Current P/L%'] > -1) & (df_tmp['Current P/L%'] < 1)].sort_values(by = 'CumlRnk', ascending=False)
df_tmp[cols].head()

,Symbol,Today P/L%,Current P/L%,FTT%,OTT%,FTT Amt,Current P/L,Current,FTT,Dev%_PE,RSI_14,Conviction,Spread%,CumlRnk,RRR Ind,CurrAlloc%,Gained%,Criteria,Strategy,Category


In [14]:
# OX40N stocks to exit
df_tmp = df_portfolio_features[(df_portfolio_features['Symbol'].isin(list_ox40n))].sort_values(by = 'Current P/L%', ascending=False)
df_tmp[cols].head()

,Symbol,Today P/L%,Current P/L%,FTT%,OTT%,FTT Amt,Current P/L,Current,FTT,Dev%_PE,RSI_14,Conviction,Spread%,CumlRnk,RRR Ind,CurrAlloc%,Gained%,Criteria,Strategy,Category
25,FINCABLES,4.55,50.97,36.74,106.44,81895.0,75260.0,222904.0,1629.94,10.09,42.0,M-SC,25.86,220.0,0.92,1.48,68.89,OX40N,ATH,CABLES
66,SIS,3.06,-11.11,36.42,21.26,35946.0,-12334.0,98698.0,528.00,89.53,47.0,H-SC,8.20,168.0,-0.34,0.66,46.75,OX40N,NTT,MISC
12,BOSCH-HCIL,0.27,-24.64,32.67,-0.01,31468.0,-31485.0,96320.0,2282.00,752.22,46.0,M-SC,20.87,231.0,-1.00,0.64,63.95,OX40N,BTT,AC
76,TTKPRESTIG,4.90,-28.87,40.72,0.09,29189.0,-29094.0,71683.0,770.00,-13.37,58.0,M-SC,4.66,250.0,-1.00,0.48,29.02,OX40N,NTT,DURABLES
49,KANSAINER,4.38,-34.81,74.07,13.47,130206.0,-93879.0,175788.0,340.00,-31.79,62.0,H-SC,8.37,162.0,-0.72,1.17,24.25,XY24,NTT,PAINTS


In [15]:
# top 5 to book for rotation
df_tmp = df_portfolio_features[(df_portfolio_features['Current P/L%'] > 20) ].sort_values(by = 'CumlRnk', ascending=False)
df_tmp[cols].head()

,Symbol,Today P/L%,Current P/L%,FTT%,OTT%,FTT Amt,Current P/L,Current,FTT,Dev%_PE,RSI_14,Conviction,Spread%,CumlRnk,RRR Ind,CurrAlloc%,Gained%,Criteria,Strategy,Category
25,FINCABLES,4.55,50.97,36.74,106.44,81895.0,75260.0,222904.0,1629.94,10.09,42.0,M-SC,25.86,220.0,0.92,1.48,68.89,OX40N,ATH,CABLES
85,ZYDUSLIFE,-4.23,25.29,8.16,35.51,21426.0,52996.0,262570.0,1285.01,23.89,54.0,H-LC,11.58,120.0,2.47,1.75,38.18,AR,ATH,PHARMA


In [16]:
# top 5 to monitor for exit
df_tmp = df_portfolio_features[~df_portfolio_features['Conviction'].isin(['X-LC','H-LC','X-MC','X-SC'])]
df_tmp = df_tmp[(df_tmp['Current P/L%'] > 0) ].sort_values(by = 'Dev%_PE', ascending=False)
df_tmp[cols].head()

,Symbol,Today P/L%,Current P/L%,FTT%,OTT%,FTT Amt,Current P/L,Current,FTT,Dev%_PE,RSI_14,Conviction,Spread%,CumlRnk,RRR Ind,CurrAlloc%,Gained%,Criteria,Strategy,Category
25,FINCABLES,4.55,50.97,36.74,106.44,81895.0,75260.0,222904.0,1629.94,10.09,42.0,M-SC,25.86,220.0,0.92,1.48,68.89,OX40N,ATH,CABLES
67,STARHEALTH,1.52,2.66,43.45,47.26,125393.0,7469.0,288592.0,761.00,-9.42,46.0,H-SC,5.19,173.0,0.06,1.92,24.78,XY24,NTT,INSURANCE


In [17]:
# top 5 to book for rotation from weak conviction
df_tmp = df_portfolio_features[~df_portfolio_features['Conviction'].isin(['X-LC','H-LC','X-MC','X-SC'])]
df_tmp = df_tmp[~df_tmp['Criteria'].isin(['XY25','XY24','X40', 'X40N'])].sort_values(by = 'RRR Ind', ascending=False)
df_tmp[cols].head()

,Symbol,Today P/L%,Current P/L%,FTT%,OTT%,FTT Amt,Current P/L,Current,FTT,Dev%_PE,RSI_14,Conviction,Spread%,CumlRnk,RRR Ind,CurrAlloc%,Gained%,Criteria,Strategy,Category
25,FINCABLES,4.55,50.97,36.74,106.44,81895.0,75260.0,222904.0,1629.94,10.09,42.0,M-SC,25.86,220.0,0.92,1.48,68.89,OX40N,ATH,CABLES
84,WIPRO,-1.74,-10.59,115.88,93.01,259261.0,-26508.0,223732.0,420.00,-24.60,66.0,M-LC,8.69,98.0,-0.10,1.49,24.08,XR,NTT,IT
80,VALIANTORG,3.27,-9.87,93.22,74.15,137460.0,-16147.0,147458.0,838.00,NaN,51.0,NaN,47.98,145.0,-0.12,0.98,119.44,XR,NTT,CHEMICALS
70,TANLA,-0.45,-25.51,188.87,115.17,380922.0,-69076.0,201685.0,1903.93,NaN,78.0,NaN,11.40,147.0,-0.18,1.34,78.29,AR,ATH,IT
42,INDIAMART,3.24,-31.62,193.82,100.92,163466.0,-38997.0,84339.0,4675.60,NaN,50.0,NaN,16.76,140.0,-0.24,0.56,7.00,AR,ATH,MISC


In [18]:
# top 5 to accumulate based on lowest RSI_14
df_tmp = df_portfolio_features[df_portfolio_features['Conviction'].isin(['X-LC','H-LC','X-MC','X-SC'])].sort_values(by = 'RSI_14', ascending=True)
df_tmp[cols].head()

,Symbol,Today P/L%,Current P/L%,FTT%,OTT%,FTT Amt,Current P/L,Current,FTT,Dev%_PE,RSI_14,Conviction,Spread%,CumlRnk,RRR Ind,CurrAlloc%,Gained%,Criteria,Strategy,Category
40,ICICIPRULI,0.66,-32.20,27.50,-13.56,36730.0,-63443.0,133562.0,522.35,-56.58,27.0,X-MC,9.01,78.0,-1.73,0.89,0.66,X40,ATH,INSURANCE
6,AWL,-0.27,-43.58,196.00,67.02,394046.0,-155265.0,201044.0,485.00,-58.47,32.0,X-SC,6.97,81.0,-0.39,1.34,0.00,XY24,NTT,FMCG
37,HONAUT,2.55,-22.71,95.21,50.88,141911.0,-43788.0,149050.0,58192.80,-39.63,32.0,X-SC,6.26,90.0,-0.31,0.99,13.56,X40N,ATH,ELECTRICAL
8,BAJAJHLDNG,0.69,-6.65,36.03,26.99,105252.0,-20803.0,292124.0,14191.91,-15.21,42.0,X-LC,3.99,28.0,-0.20,1.94,21.47,X40,ATH,FINANCE
27,GLAXO,-1.68,1.64,33.51,35.69,67872.0,3260.0,202544.0,3380.16,-23.72,45.0,X-MC,10.12,56.0,0.05,1.35,20.38,X40,ATH,PHARMA


In [19]:
# top 5 to accumulate based on Spread and CumlRnk
df_tmp = df_portfolio_features[df_portfolio_features['Conviction'].isin(['X-LC','H-LC','X-MC','X-SC'])]
df_tmp = df_tmp[(df_tmp['CumlRnk'] < 100)].sort_values(by = 'Spread%', ascending=True)
df_tmp[cols].head()

,Symbol,Today P/L%,Current P/L%,FTT%,OTT%,FTT Amt,Current P/L,Current,FTT,Dev%_PE,RSI_14,Conviction,Spread%,CumlRnk,RRR Ind,CurrAlloc%,Gained%,Criteria,Strategy,Category
1,ABBOTINDIA,5.12,-9.02,26.81,15.37,43625.0,-16135.0,162720.0,34390.26,-25.92,62.0,X-MC,1.44,58.0,-0.37,1.08,10.22,X40,ATH,PHARMA
55,PAGEIND,1.33,-6.76,42.48,32.85,106306.0,-18129.0,250250.0,50936.07,-30.68,51.0,X-MC,1.66,55.0,-0.17,1.66,19.50,X40,ATH,APPARELS
17,CIPLA,0.37,5.39,17.56,23.89,55639.0,16197.0,316852.0,1648.14,19.62,65.0,X-LC,1.92,18.0,0.29,2.11,18.68,X40N,ATH,PHARMA
11,BERGEPAINT,1.13,-10.60,40.18,25.32,92584.0,-27327.0,230422.0,680.00,-22.28,62.0,X-MC,2.21,68.0,-0.30,1.53,20.83,XY24,NTT,PAINTS
8,BAJAJHLDNG,0.69,-6.65,36.03,26.99,105252.0,-20803.0,292124.0,14191.91,-15.21,42.0,X-LC,3.99,28.0,-0.20,1.94,21.47,X40,ATH,FINANCE


In [20]:
# top 5 to accumulate based on lowest Gained%
df_tmp = df_portfolio_features[df_portfolio_features['Conviction'].isin(['X-LC','H-LC','X-MC','X-SC'])].sort_values(by = 'Gained%', ascending=True)
df_tmp[cols].head()

,Symbol,Today P/L%,Current P/L%,FTT%,OTT%,FTT Amt,Current P/L,Current,FTT,Dev%_PE,RSI_14,Conviction,Spread%,CumlRnk,RRR Ind,CurrAlloc%,Gained%,Criteria,Strategy,Category
6,AWL,-0.27,-43.58,196.00,67.02,394046.0,-155265.0,201044.0,485.00,-58.47,32.0,X-SC,6.97,81.0,-0.39,1.34,0.00,XY24,NTT,FMCG
40,ICICIPRULI,0.66,-32.20,27.50,-13.56,36730.0,-63443.0,133562.0,522.35,-56.58,27.0,X-MC,9.01,78.0,-1.73,0.89,0.66,X40,ATH,INSURANCE
35,HDFCLIFE,1.33,-7.76,52.38,40.56,150340.0,-24141.0,287018.0,811.41,-32.80,50.0,X-LC,10.22,31.0,-0.16,1.91,3.68,X40,ATH,INSURANCE
28,GODREJCP,2.91,-22.77,71.04,32.10,136910.0,-56814.0,192723.0,1452.16,-24.23,51.0,X-MC,16.80,32.0,-0.41,1.28,4.42,X40N,ATH,FMCG
56,PGHH,-1.38,-47.85,151.25,31.02,168408.0,-102176.0,111344.0,17484.41,-61.63,47.0,X-SC,24.66,57.0,-0.61,0.74,4.82,X40,ATH,FMCG


In [21]:
# top 5 to accumulate based on lowest CurrAlloc%
df_tmp = df_portfolio_features[df_portfolio_features['Conviction'].isin(['X-LC','H-LC','X-MC','X-SC'])].sort_values(by = 'CurrAlloc%', ascending=True)
df_tmp[cols].head()

,Symbol,Today P/L%,Current P/L%,FTT%,OTT%,FTT Amt,Current P/L,Current,FTT,Dev%_PE,RSI_14,Conviction,Spread%,CumlRnk,RRR Ind,CurrAlloc%,Gained%,Criteria,Strategy,Category
56,PGHH,-1.38,-47.85,151.25,31.02,168408.0,-102176.0,111344.0,17484.41,-61.63,47.0,X-SC,24.66,57.0,-0.61,0.74,4.82,X40,ATH,FMCG
40,ICICIPRULI,0.66,-32.20,27.50,-13.56,36730.0,-63443.0,133562.0,522.35,-56.58,27.0,X-MC,9.01,78.0,-1.73,0.89,0.66,X40,ATH,INSURANCE
2,ACC,1.30,-46.95,229.54,74.81,326489.0,-125892.0,142236.0,3906.00,-47.20,52.0,X-SC,16.34,85.0,-0.39,0.95,5.37,XY24,BTT,CEMENT
37,HONAUT,2.55,-22.71,95.21,50.88,141911.0,-43788.0,149050.0,58192.80,-39.63,32.0,X-SC,6.26,90.0,-0.31,0.99,13.56,X40N,ATH,ELECTRICAL
82,VOLTAS,1.80,-16.70,74.31,45.20,118688.0,-32022.0,159720.0,1856.00,14.67,54.0,X-MC,20.87,79.0,-0.27,1.06,12.53,XY25,NTT,AC


In [22]:
# top 5 to accumulate based on CumlRnk
df_tmp = df_portfolio_features.sort_values(by = 'CumlRnk', ascending=True)
df_tmp[cols].head()

,Symbol,Today P/L%,Current P/L%,FTT%,OTT%,FTT Amt,Current P/L,Current,FTT,Dev%_PE,RSI_14,Conviction,Spread%,CumlRnk,RRR Ind,CurrAlloc%,Gained%,Criteria,Strategy,Category
47,ITC,2.23,-20.27,61.43,28.70,214317.0,-88715.0,348880.0,452.00,-27.90,67.0,X-LC,8.28,1.0,-0.41,2.32,9.80,X40,NTT,FMCG
72,TCS,-2.52,-26.63,72.76,26.75,260548.0,-129983.0,358092.0,4208.34,-40.62,58.0,X-LC,4.96,2.0,-0.50,2.38,24.20,X40,ATH,IT
45,INFY,-1.73,-18.38,74.79,42.66,244696.0,-73677.0,327178.0,1972.00,-40.86,57.0,X-LC,8.96,3.0,-0.30,2.17,17.13,X40,NTT,IT
74,TMPV,0.65,-32.21,115.44,46.05,350113.0,-144086.0,303286.0,600.00,-87.07,49.0,X-LC,17.67,4.0,-0.41,2.02,6.62,XY24,NTT,AUTO
33,HCLTECH,-2.37,-3.86,30.75,25.70,106713.0,-13938.0,347035.0,1837.07,-8.41,62.0,X-LC,4.76,8.0,-0.13,2.31,38.46,X40,ATH,IT


In [23]:
# top 5 for average up
df_tmp = df_portfolio_features[(df_portfolio_features['Dev%_200'] > 5)].sort_values(by = 'CurrAlloc%', ascending=True)
df_tmp[cols].head()

,Symbol,Today P/L%,Current P/L%,FTT%,OTT%,FTT Amt,Current P/L,Current,FTT,Dev%_PE,RSI_14,Conviction,Spread%,CumlRnk,RRR Ind,CurrAlloc%,Gained%,Criteria,Strategy,Category
12,BOSCH-HCIL,0.27,-24.64,32.67,-0.01,31468.0,-31485.0,96320.0,2282.00,752.22,46.0,M-SC,20.87,231.0,-1.00,0.64,63.95,OX40N,BTT,AC
80,VALIANTORG,3.27,-9.87,93.22,74.15,137460.0,-16147.0,147458.0,838.00,NaN,51.0,NaN,47.98,145.0,-0.12,0.98,119.44,XR,NTT,CHEMICALS
70,TANLA,-0.45,-25.51,188.87,115.17,380922.0,-69076.0,201685.0,1903.93,NaN,78.0,NaN,11.40,147.0,-0.18,1.34,78.29,AR,ATH,IT
25,FINCABLES,4.55,50.97,36.74,106.44,81895.0,75260.0,222904.0,1629.94,10.09,42.0,M-SC,25.86,220.0,0.92,1.48,68.89,OX40N,ATH,CABLES
85,ZYDUSLIFE,-4.23,25.29,8.16,35.51,21426.0,52996.0,262570.0,1285.01,23.89,54.0,H-LC,11.58,120.0,2.47,1.75,38.18,AR,ATH,PHARMA


In [24]:
# top 5 RSP
df_tmp = df_portfolio_features.sort_values(by = 'RSP', ascending=False)
df_tmp[cols].head()

,Symbol,Today P/L%,Current P/L%,FTT%,OTT%,FTT Amt,Current P/L,Current,FTT,Dev%_PE,RSI_14,Conviction,Spread%,CumlRnk,RRR Ind,CurrAlloc%,Gained%,Criteria,Strategy,Category
70,TANLA,-0.45,-25.51,188.87,115.17,380922.0,-69076.0,201685.0,1903.93,NaN,78.0,NaN,11.40,147.0,-0.18,1.34,78.29,AR,ATH,IT
71,TATAELXSI,1.03,-45.88,143.97,32.03,102715.0,-60490.0,71345.0,9161.00,-38.60,61.0,H-SC,14.10,158.0,-0.59,0.47,25.33,OX40N,NTT,IT
84,WIPRO,-1.74,-10.59,115.88,93.01,259261.0,-26508.0,223732.0,420.00,-24.60,66.0,M-LC,8.69,98.0,-0.10,1.49,24.08,XR,NTT,IT
33,HCLTECH,-2.37,-3.86,30.75,25.70,106713.0,-13938.0,347035.0,1837.07,-8.41,62.0,X-LC,4.76,8.0,-0.13,2.31,38.46,X40,ATH,IT
72,TCS,-2.52,-26.63,72.76,26.75,260548.0,-129983.0,358092.0,4208.34,-40.62,58.0,X-LC,4.96,2.0,-0.50,2.38,24.20,X40,ATH,IT


In [25]:
# Top N allocation
df_tmp = df_portfolio_features.sort_values(by = 'CurrAlloc%', ascending=False)
top_n_values = [10, 25, 50]

sum_df = pd.DataFrame({
    'Top_N': top_n_values,
    'Sum_Alloc%': [df_tmp['CurrAlloc%'].head(n).sum() for n in top_n_values]
})

sum_df

,Top_N,Sum_Alloc%
0,10,22.14
1,25,49.33
2,50,80.41


In [26]:
# market-cap-wise allocation
df_tmp = df_portfolio_features[cols]
df_tmp.groupby(df_tmp['Conviction'].str[-2:])['CurrAlloc%'].sum().sort_values(ascending=False)

Conviction
LC    42.54
MC    20.67
SC    20.11
Name: CurrAlloc%, dtype: float64

In [27]:
# criteria-wise allocation
df_portfolio_features.groupby('Criteria')['CurrAlloc%'].sum().sort_values(ascending=False)

Criteria
X40      27.36
XY24     22.32
X40N     18.61
XR        9.84
XY25      9.50
OX40N     6.60
AR        5.01
SR        0.75
Name: CurrAlloc%, dtype: float64

In [28]:
# conviction-wise allocation
df_portfolio_features.groupby('Conviction')['CurrAlloc%'].sum().sort_values(ascending=False)

Conviction
X-LC    39.30
X-MC    18.32
H-SC     8.03
X-SC     7.04
M-SC     5.04
H-LC     1.75
M-LC     1.49
M-MC     1.05
H-MC     0.97
L-MC     0.33
Name: CurrAlloc%, dtype: float64

In [29]:
# sector-wise stats
df_tmp = df_portfolio_features.groupby('Category')[['CurrAlloc%', 'Current', 'Current P/L', 'FTT Amt']].sum().sort_values(by=['Current', 'Current P/L'], ascending=False)
df_tmp['Current P/L%'] = round(df_tmp['Current P/L'] * 100 / df_tmp['Current'], 2)
df_tmp['FTT%'] = round(df_tmp['FTT Amt'] * 100 / df_tmp['Current'], 2)
cols = ['CurrAlloc%', 'Current P/L%', 'FTT%']
df_tmp[cols].sort_values(by=['CurrAlloc%'], ascending=False)

,CurrAlloc%,Current P/L%,FTT%
Category,,,
FMCG,17.89,-27.44,73.21
IT,12.88,-32.28,101.42
FINANCE,10.49,-27.79,82.38
AUTO,8.28,-20.17,63.05
PHARMA,6.29,5.96,19.96
INSURANCE,5.84,-12.64,43.28
MISC,5.42,-51.07,111.79
PAINTS,5.34,-28.64,51.21
BANKS,4.96,-20.24,80.55


In [30]:
# money to be made criteria-wise
df_portfolio_features.groupby('Criteria')['FTT Amt'].agg(['sum', 'count']).sort_values(by='sum', ascending=False)

,sum,count
Criteria,,
XY24,4080451.0,19
X40,2274332.0,17
XR,1574513.0,13
X40N,1535958.0,12
AR,1038874.0,7
OX40N,796951.0,10
XY25,784950.0,6
SR,299889.0,2


In [31]:
# money to be made conviction-wise
df_portfolio_features.groupby('Conviction')['FTT Amt'].agg(['sum', 'count']).sort_values(by='sum', ascending=False)

,sum,count
Conviction,,
X-LC,3030299.0,19
X-MC,1647026.0,14
X-SC,1215832.0,6
H-SC,942856.0,8
M-SC,405228.0,6
H-MC,266994.0,1
M-LC,259261.0,1
M-MC,87449.0,1
L-MC,53289.0,1


In [32]:
# money to be made criteria and conviction-wise
df_portfolio_features.groupby(['Conviction', 'Criteria'])['FTT Amt'].agg(['sum', 'count']).sort_values(by='sum', ascending=False)

sum  count
Conviction Criteria                  
X-LC       X40       1355037.0      8
X-SC       XY24       720535.0      2
X-LC       X40N       597712.0      5
           XY24       582704.0      2
H-SC       XY24       580068.0      3
X-MC       X40N       545561.0      4
           X40        528558.0      6
X-LC       XY25       382086.0      3
X-MC       XY25       315415.0      2
X-SC       X40        269322.0      2
H-MC       XY24       266994.0      1
M-LC       XR         259261.0      1
X-MC       XY24       257492.0      2
H-SC       OX40N      249754.0      3
X-SC       X40N       225975.0      2
M-SC       OX40N      210523.0      4
           XY24       135154.0      1
H-SC       XR         113034.0      2
X-LC       XR         112760.0      1
M-MC       XY25        87449.0      1
M-SC       XR          59551.0      1
L-MC       XR          53289.0      1
H-LC       AR          21426.0      1

In [33]:
# notebook execution time

end_time = time.time()
execution_time = round(end_time - start_time, 0)
print(f"Notebook execution time: {execution_time} seconds")

Notebook execution time: 23.0 seconds
